# Ответы на RL вопросы по неделям

## Прокофьев Кирилл 3825М1ФИ1

## Неделя 1

### 1. Базовые понятия RL (по материалам презентации)

**Ключевое отличие от обучения с учителем:**  
В supervised learning у нас есть размеченные данные $(x, y)$ и мы минимизируем функцию потерь $L(y, a_\theta(x))$. 
В RL **нет эталонных ответов** — агент учится через взаимодействие со средой, получая лишь скалярную обратную связь (награду) за свои действия.

**Основные компоненты:**
- **Агент** — обучающийся субъект, принимающий решения.
- **Среда (Environment)** — внешний мир, с которым агент взаимодействует. Может быть полностью или частично наблюдаемой.
- **Состояние $s \in \mathcal{S}$** — описание ситуации, в которой находится агент.
- **Действие $a \in \mathcal{A}$** — выбор агента в данном состоянии.
- **Награда $r \in \mathbb{R}$** — мгновенная оценка качества перехода.
- **Политика $\pi(a|s)$** — стратегия агента, задающая распределение вероятностей действий: $\pi(a|s) = P(a_t = a \mid s_t = s)$.

**MDP-формализм (Markov Decision Process):**  
Среда описывается кортежем $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R})$, где:
- Динамика переходов: $P(s_{t+1} \mid s_t, a_t)$
- **Марковское предположение:** будущее зависит только от текущего состояния и действия:
  $$P(s_{t+1} \mid s_t, a_t, s_{t-1},a_{t-1}) = P(s_{t+1} \mid s_t, a_t)$$

**Целевая функция:**  
Агент стремится максимизировать ожидаемую суммарную награду за эпизод:
$$R = \sum_{t} r_t, \quad \text{найти } \pi^* = \arg\max_\pi \mathbb{E}_\pi[R]$$

**Фундаментальные проблемы:**
- **Exploration vs Exploitation:** если всегда выбирать текущее «оптимальное» действие, можно никогда не обнаружить лучшую стратегию.
- **Credit assignment:** какие именно действия привели к полученной награде?
- **Разреженность наград:** обратная связь может приходить редко (например, только в конце игры).

**Упрощённый случай — бандиты (Bandits):**  
Задача без динамики состояний: агент видит контекст (features пользователя), выбирает действие (показать баннер) и получает награду (клик). Это «одношаговый» RL, полезный для A/B-тестирования и рекомендаций.

---

### 2. Метод кросс-энтропии (CEM)

**Общая идея (принцип «естественного отбора» траекторий):**
1. Инициализировать политику $\pi$.
2. Повторять цикл:
   - Сгенерировать $N$ эпизодов (сессий) при текущей политике.
   - Отобрать $M$ лучших по суммарной награде — **элитные траектории**.
   - Обновить политику так, чтобы она чаще воспроизводила действия из элиты.

**Табличный вариант:**  
Политика хранится как матрица $\pi(a|s) = A[s, a]$. После отбора элиты новые вероятности вычисляются как частоты действий в элитных сессиях для каждого состояния:
$$\pi ( a | s ) = \frac { \sum _ { s _ { t } , a _ { t } \in E l i t e } \left[ s _ { t } = s \right] \left[ a _ { t } = a \right] } { \sum _ { s _ { t } , a _ { t } \in E l i t e } \left[ s _ { t } = s \right] }$$
Метод прост и интерпретируем, но масштабируется только на маленькие дискретные пространства состояний.

**Приближённый метод кросс-энтропии (Approximate CEM) с нейросетью** 

Когда пространство состояний велико, непрерывно или содержит визуальные данные, хранить политику в виде явной матрицы $\pi(a|s)$ невозможно. Презентация предлагает заменить её параметрической моделью $\pi_W(a|s)$, где $W$ — веса нейронной сети.

Вместо пересчёта эмпирических частот (как в табличном варианте) задача сводится к **максимизации логарифмического правдоподобия** действий, взятых из элитных траекторий:
$$W^* = \arg\max_W \sum_{(s_i, a_i) \in \text{Elite}} \log \pi_W(a_i \mid s_i)$$

Обновление весов происходит через градиентный подъём:
$$W_{i+1} = W_i + \alpha \nabla_W \left[ \sum_{(s_i, a_i) \in \text{Elite}} \log \pi_{W_i}(a_i \mid s_i) \right]$$
где $\alpha$ — learning rate, а градиент вычисляется по всем парам «состояние–действие», попавшим в элиту.

Архитектура под тип пространства действий:
- **Дискретные действия (`MLPClassifier`):** Сеть выдаёт логиты для каждого действия, к которым применяется Softmax. Оптимизация эквивалентна минимизации кросс-энтропии между предсказанным распределением и one-hot вектором элитного действия.
- **Непрерывные действия (`MLPRegressor`):** Политика задаётся гауссовым распределением:
  $$\pi_W(a \mid s) = \mathcal{N}\big(\mu_W(s), \sigma^2\big)$$
  Сеть предсказывает среднее $\mu(s)$, а дисперсия $\sigma^2$ может быть фиксированным гиперпараметром или выходом отдельной ветви сети. Градиент считается по логарифму плотности нормального распределения. Структура алгоритма при этом почти не меняется.

Цикл обучения: 
1. Инициализировать веса $W_0 \leftarrow \text{random}$.
2. Сгенерировать $N$ эпизодов, сэмплируя действия из $\pi_{W_i}(a|s)$.
3. Вычислить суммарную награду каждого эпизода и отобрать $M$ лучших $\to$ `Elite`.
4. Сконкатенировать все переходы $(s_t, a_t)$ из элиты в обучающую выборку.
5. Выполнить шаг оптимизации: `nn.fit(elite_states, elite_actions)` или одно обновление по формуле выше.
6. Повторять, пока политика не стабилизируется.

Приближённый CEM превращает RL-задачу в последовательность задач обучения с учителем на самогенерированных данных. Метод не требует вычисления TD-ошибок, функций ценности или градиентов среды, что делает его устойчивым и простым в реализации, но чувствительным к качеству отбора элиты и объёму сэмплируемых траекторий.


---

## Неделя 2

### 1. $V(s)$ и $Q(s, a)$

Это функции ценности, которые количественно оценивают ожидаемую долгосрочную полезность состояний и действий при следовании определённой политике $\pi$.

**$V_\pi(s)$ — функция ценности состояния (State-value function)**  
Ожидаемый суммарный дисконтированный возврат, если агент начинает в состоянии $s$ и далее строго следует политике $\pi$:
$$V_\pi(s) = \mathbb{E}_\pi \left[ G_t \mid s_t = s \right]$$
 
Связь с динамикой среды даётся уравнением Беллмана для ожиданий:
$$V_\pi(s) = \sum_{a} \pi(a|s) \sum_{r, s'} p(r, s'|s, a) \left[ r + \gamma V_\pi(s') \right]$$

**$Q_\pi(s, a)$ — функция ценности действия (Action-value function)**  
Ожидаемый суммарный возврат, если агент принудительно выполняет действие $a$ в состоянии $s$, а затем возвращается к политике $\pi$:
$$Q_\pi(s, a) = \mathbb{E}_\pi \left[ G_t \mid s_t = s, a_t = a \right]$$
 
Уравнение Беллмана:
$$Q_\pi(s, a) = \sum_{r, s'} p(r, s'|s, a) \left[ r + \gamma \sum_{a'} \pi(a'|s') Q_\pi(s', a') \right]$$

**Взаимосвязь:** $V_\pi(s) = \sum_a \pi(a|s) Q_\pi(s, a)$. Функция $Q$ содержит больше информации, так как позволяет напрямую выбирать оптимальные действия без знания модели переходов $p(r, s'|s, a)$.

---

### 2. Метод Policy Iteration

Policy Iteration — это канонический алгоритм динамического программирования для точного решения MDP. Он реализует принцип **Generalized Policy Iteration (GPI)**, строго чередуя два этапа: оценку текущей стратегии и её жадное улучшение. Алгоритм опирается на уравнения Беллмана и гарантирует сходимость к оптимальной политике $\pi^*$ за конечное число шагов в конечных MDP.


#### 1. Policy Evaluation (Оценка политики) 
*Задача:* Для фиксированной политики $\pi$ вычислить функцию ценности состояния $v_\pi(s)$.

**Математическая основа:**  
Оценка строится на уравнении Беллмана для ожиданий:
$$v_\pi(s) = \sum_{a} \pi(a|s) \sum_{r, s'} p(r, s'|s, a) \left[ r + \gamma v_\pi(s') \right]$$
Это система линейных уравнений, где число неизвестных равно числу уравнений и равно числу состояний $|\mathcal{S}|$.

**Алгоритм оценки:**  
Начиная с произвольного $v_0(s)$, итеративно применяются обновления Беллмана:
$$v_{k+1}(s) \leftarrow \sum_{a} \pi(a|s) \sum_{r, s'} p(r, s'|s, a) \left[ r + \gamma v_k(s') \right]$$
Процесс повторяется, пока $\max_s |v_{k+1}(s) - v_k(s)| < \epsilon$.



#### 2. Policy Improvement (Улучшение политики)
*Идея:* Имея оценённую $v_\pi(s)$, можно явно вычислить ценность действий $q_\pi(s, a)$ и построить новую политику, действующую жадно относительно этих оценок.

**Формула обновления:**
$$\pi'(s) = \arg\max_a q_\pi(s, a) = \arg\max_a \sum_{r, s'} p(r, s'|s, a) \left[ r + \gamma v_\pi(s') \right]$$

**Теоретические гарантии:**
- **Гарантия улучшения:** Новая политика $\pi'$ всегда не хуже старой: $v_{\pi'}(s) \geq v_\pi(s)$ для всех $s \in \mathcal{S}$.
- **Условие остановки:** Если после шага улучшения новая политика совпадает со старой ($\pi' = \pi$), процесс завершается. Это означает, что $\pi$ удовлетворяет уравнению Беллмана для оптимальности и является $\pi^*$.



#### 3. Полный цикл Policy Iteration
Алгоритм работает по детерминированной схеме:
1. Инициализировать политику $\pi_0$ (например, равномерно случайную).
2. **Policy Evaluation:** Итеративно вычислять $v_{\pi_k}$ уравнением Беллмана до достижения точности $\epsilon$.
3. **Policy Improvement:** Обновить политику жадно: $\pi_{k+1}(s) = \arg\max_a q_{\pi_k}(s, a)$.
4. Если $\pi_{k+1} = \pi_k$, остановиться. Иначе перейти к шагу 2.

Цикл строго монотонен и сходится к оптимальной стратегии, так как пространство детерминированных политик конечно, а каждый шаг улучшения либо повышает ценность, либо достигает оптимума.



#### 4. Вычислительная сложность 

- **Сложность одной итерации PI:** $O(|\mathcal{A}||\mathcal{S}|^2 + |\mathcal{S}|^3)$. Затратнее из-за необходимости доводить оценку до сходимости (фактически решать систему линейных уравнений).

---

## Неделя 3

### 1. Q-learning vs SARSA

Оба алгоритма относятся к **model-free temporal difference** методам и обучаются непосредственно на семплированных траекториях $(s, a, r, s')$, не требуя знания динамики среды $P(s'|s,a)$. Они используют общую схему обновления с коэффициентом обучения $\alpha$:
$$Q(s_t, a_t) \leftarrow \alpha \cdot \hat{Q}(s_t, a_t) + (1 - \alpha) Q(s_t, a_t)$$
Ключевое различие заключается в способе формирования целевого значения $\hat{Q}(s, a)$.

**Q-learning (Off-policy)**  
Целевое значение строится на основе *теоретически лучшего* действия в следующем состоянии:
$$\hat{Q}(s, a) = r(s, a) + \gamma \cdot \max_{a'} Q(s', a')$$
Алгоритм игнорирует стратегию исследования и напрямую аппроксимирует оптимальную функцию ценности $Q^*(s,a)$. Это позволяет обучать оптимальную политику даже если агент совершает случайные действия, но полученная стратегия не учитывает риск исследовательских шагов во время выполнения.

**SARSA (On-policy)**  
Целевое значение формируется на основе *фактически выбранного* действия $a'$ в следующем состоянии (которое семплируется из текущей политики $\pi$):
$$\hat{Q}(s, a) = r(s, a) + \gamma \cdot Q(s', a')$$
Название происходит из последовательности семпла: **S**tate, **A**ction, **R**eward, **S**tate, **A**ction. SARSA обучается оценивать ценность *текущей поведенческой политики*, включая её стохастичность (например, $\epsilon$-greedy Exploration). 

**Практическое различие:**  
В среде с обрывом Q-learning находит кратчайший путь к цели, но агент часто падает, так как при $\epsilon$-greedy исследовании случайно делает шаг к краю. SARSA, учитывая вероятность случайных действий, обучается более безопасной траектории, держащейся дальше от обрыва. Q-learning оптимален *без учёта exploration*, SARSA оптимален *с учётом exploration*.

---

### 2. Exploration vs Exploitation

Это фундаментальный компромисс между использованием накопленных знаний (Exploitation) и поиском новых, потенциально более выгодных действий (Exploration).

**Проблема чистого Exploitation:**  
Если агент всегда выбирает $\arg\max_a Q(s,a)$, он может застрять в субоптимальном поведении.

**Стратегии баланса:**
- **$\epsilon$-greedy:** с вероятностью $\epsilon$ выбирается случайное действие, иначе — действие с максимальным $Q$-значением.
- **Softmax:** вероятность действия пропорциональна экспоненте нормализованной ценности:
  $$\pi(a|s) = \text{softmax}\left(\frac{Q(s, a)}{\tau}\right)$$
  где $\tau$ (temperature) контролирует степень исследования.

**Затухание исследования:**  
Для гарантии сходимости к оптимальной политике вероятность исследования должна плавно снижаться со временем ($\epsilon \to 0$), превращая стратегию в *greedy in the limit*. Однако в нестационарных средах полное затухание может привести к потере адаптивности.

**Связь с алгоритмами:**  
Выбор стратегии exploration напрямую влияет на то, какую политику выучит агент. Q-learning стремится к теоретическому оптимуму, игнорируя шум exploration, тогда как SARSA и Expected SARSA встраивают распределение exploration в оценку ценности, что делает их поведение более предсказуемым и безопасным в реальных условиях.

---


## Неделя 4

### 1. DQN (Deep Q-Network)
Переход от табличного хранения $Q(s, a)$ к аппроксимации параметрической функцией (нейронной сетью) с весами $\theta$. Это необходимо, так как в реальных задачах пространство состояний часто огромно или непрерывно, и явная таблица неприменима.

**Архитектура:** Сеть принимает состояние $s$ (например, стек последних кадров) и за один прямой проход предсказывает вектор $Q$-значений для всех допустимых действий. Выходной слой линейный (без нелинейности), так как он должен предсказывать непрерывные величины.

**Обучение:** Задача сводится к регрессии. Параметры обновляются градиентным спуском по MSE между текущим предсказанием и TD-целью:
$$L(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[ \left( Q(s, a; \theta) - y \right)^2 \right]$$
где:
$y = r + \gamma \max_{a'} Q(s', a'; \theta)$

Во время взаимодействия со средой действия выбираются по $\epsilon$-greedy правилу.

---

### 2. Experience Replay
**Проблема:** Последовательные переходы $(s_t, a_t, r_t, s_{t+1})$ сильно коррелируют во времени. Это нарушает предположение о независимости и одинаковом распределении (i.i.d.), необходимое для стабильной работы стохастического градиентного спуска. Кроме того, агент быстро «забывает» состояния, которые не посещал в последних шагах.

**Решение:** Все взаимодействия сохраняются в кольцевой буфер (replay buffer). Обучение происходит не на последних переходах, а на случайных мини-батчах, равномерно сэмплируемых из накопленной истории.

**Ключевые свойства:**
* Разрывает временные корреляции, приближая распределение обучающих данных к i.i.d.
* Позволяет многократно использовать один переход для обучения.
* Совместим **только с off-policy алгоритмами**, так как батчи содержат данные, собранные под старыми (более слабыми) версиями политики.

---

### 3. Target Network
**Проблема:** В базовом DQN целевое значение $y = r + \gamma \max_{a'} Q(s', a'; \theta)$ вычисляется той же сетью, которую мы обучаем. Предсказание и цель коррелируют и одновременно меняются на каждом шаге градиентного спуска, что вызывает «убегающую цель» и нестабильность обучения.

**Решение:** Вводится вторая копия сети с фиксированными параметрами $\theta^-$, которая используется исключительно для вычисления TD-target:
$$y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$$
Основная сеть $\theta$ обучается относительно этой замороженной цели.

**Обновление целевой сети:**
* **Hard update:** Жёсткое копирование весов раз в $N$ шагов: $\theta^- \leftarrow \theta$.
* **Soft update:** Плавное обновление на каждом шаге: $\theta^- \leftarrow (1-\alpha)\theta^- + \alpha\theta$

Это стабилизирует оптимизацию, превращая задачу в обучение с фиксированными (на коротком горизонте) метками.

---

### 4. Double DQN
**Проблема переоценки:** Оператор $\max$ в целевом значении систематически завышает оценку $Q$-значений, так как $\mathbb{E}[\max X_i] \geq \max \mathbb{E}[X_i]$. Ошибка аппроксимации накапливается и распространяется назад по цепочке Беллмана, что приводит к выбору субоптимальных действий.

**Решение:** Разделение процесса выбора действия и оценки его ценности. Основная сеть $\theta$ отвечает за выбор наилучшего действия в следующем состоянии, а целевая сеть $\theta^-$ — только за оценку значения этого действия.

**Формула целевого значения:**
$$y = r + \gamma Q^{A}(s^{\prime}, \operatorname*{argmax}_{a} Q^{B}(s^{\prime}, a^{\prime}))$$

**Результат:** Устраняет optimistic bias, делает оценки $Q$-значений более точными и значительно повышает медианную и среднюю производительность агента на тестовых средах по сравнению с классическим DQN.

---


## Неделя 6

### 1. Policy Gradient

**Основная идея:** Вместо аппроксимации функции ценности (что часто является более сложной задачей), мы напрямую оптимизируем параметризованную политику $\pi_\theta(a|s)$, максимизируя ожидаемый дисконтированный возврат $J(\theta)$.

**Целевая функция:**
$$J(\theta) = \mathbb{E}_{s \sim p(s), a \sim \pi_\theta(a|s)}[R(s,a)]$$

**Логарифмический трюк (Log-Derivative Trick):**  
Используя тождество $\nabla_\theta \log \pi_\theta(a|s) = \frac{\nabla_\theta \pi_\theta(a|s)}{\pi_\theta(a|s)}$, градиент целевой функции можно выразить как матожидание:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot Q(s,a) \right]$$

**Ключевые свойства:**
*   **On-policy:** Градиент вычисляется по действиям, семплированным из текущей политики.
*   **Непрерывные действия:** Политика задаётся параметрическим распределением (например, категориальным для дискретных действий или нормальным $\mathcal{N}(\mu_\theta(s), \sigma^2)$ для непрерывных), что делает метод универсальным.
*   **Встроенное исследование:** Стохастичность выборки $a \sim \pi_\theta(a|s)$ автоматически обеспечивает exploration без дополнительных эвристик вроде $\epsilon$-greedy.
*   **Сравнение с Supervised Learning:** Формула градиента идентична градиенту log-likelihood, но вместо фиксированных меток $y$ используется скаляр $Q(s,a)$, который оценивает, насколько хорошим было действие.

---

### 2. REINFORCE

**Суть:** Базовый Monte-Carlo алгоритм Policy Gradient. Поскольку истинная $Q(s,a)$ неизвестна, она заменяется на фактический дисконтированный возврат $G_t$, посчитанный по завершению эпизода.

**Алгоритм:**
1. Собрать $N$ эпизодов при текущей политике $\pi_\theta$.
2. Для каждого шага вычислить $G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$.
3. Обновить параметры:
   $$\theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t$$

**Проблема высокой дисперсии:**  
Оценка $G_t$ сильно шумит, так как зависит от всей случайности траектории. Вводим **базовую линию (baseline)** $b(s)$, которая не зависит от действия $a$:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot (Q(s,a) - b(s)) \right]$$
Вычитание $b(s)$ не меняет матожидание градиента (так как $\mathbb{E}_{a}[\nabla \log \pi_\theta(a|s)] = 0$), но снижает дисперсию, если $b(s)$ коррелирует с $Q(s,a)$. На практике в качестве $b(s)$ используют оценку $V(s)$ или скользящее среднее наград.

**Особенности:** Несмещённая оценка, требует полных эпизодов для обновления, чувствительна к выбору learning rate.

---

### 3. Actor-Critic

**Архитектура:** Гибридный метод, объединяющий прямую оптимизацию политики (Actor) и оценку ценности состояний (Critic) для снижения дисперсии и перехода к пошаговому обучению.

**Компоненты:**
*   **Critic:** Сеть $V_\theta(s)$, оценивающая, насколько выгодно находиться в состоянии $s$. Обучается минимизацией TD-ошибки:
  $$L_{critic} = \left( V_\theta(s) - [r + \gamma V_\theta(s')] \right)^2$$
*   **Actor:** Сеть $\pi_\theta(a|s)$, обновляемая градиентом политики, но вместо $G_t$ использует оценку преимущества (Advantage), полученную от Critic.

**Advantage Function:**  
Вместо полного возврата используется TD-разность как аппроксимация преимущества действия:
$$A(s,a) \approx r + \gamma V_\theta(s') - V_\theta(s)$$
Эта величина показывает, насколько выбранное действие лучше среднего ожидания для данного состояния.

**Обновление Actor:**
$$\theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta(a|s) \cdot A(s,a)$$

**Преимущества:**
*   **Низкая дисперсия:** Критик сглаживает шум наград, предоставляя стабильный сигнал.
*   **Online-обучение:** Не нужно ждать конца эпизода; обновление возможно на каждом шаге $t$.
*   **Гибкость:** Ошибки критика менее критичны, чем в value-based методах (actor всё равно будет сходиться, но медленнее). Часто дополняется энтропийной регуляризацией для предотвращения premature convergence.
*   Является фундаментом для современных алгоритмов (A2C, A3C, PPO, SAC).

---


## Неделя 7

### 1. Применение RNN для генерации последовательностей

**Encoder-Decoder RNN** архитектура, которая читает входные данные (текст, изображение) и генерирует выходную последовательность посимвольно (или по токенам).

Принцип работы Encoder-Decoder RNN: 


Входная последовательность $\to$ Encoder $\to$ Контекстный вектор h $\to$ Decoder $\to$ Выходная последовательность

**Обучение vs Инференс:**
*   **Supervised обучение:** Модель максимизирует логарифмическое правдоподобие эталонных последовательностей из датасета. Градиент вычисляется как:
$$\nabla \text{llh} = \mathbb{E}_{x, y_{opt} \sim D} \left[ \nabla \log P_\theta(y_{opt} \mid x) \right]$$
На каждом шаге декодеру подаются *идеальные* предыдущие токены $y_{0:t}$ из разметки (teacher forcing).
*   **Инференс:** Модель использует собственные предсказания $\hat{y}_{0:t}$ в качестве входа для следующего шага. Обычно выбирается жадный поиск ($\arg\max$) или сэмплирование.

**Проблема Distribution Shift (Exposure Bias):**
Во время обучения модель видит распределение $P(y_{t+1} \mid x, y_{0:t})$, а во время генерации вынуждена работать с $P(y_{t+1} \mid x, \hat{y}_{0:t})$. Если модель допускает ошибку на раннем шаге, её внутренние представления отклоняются от тренировочных, что приводит к каскадным ошибкам и нестабильной генерации. Кроме того, для многих задач существует несколько корректных вариантов перевода/ответа, а supervised learning пытается усреднить все возможные правильные пути, что размывает вероятностную массу модели.

---

### 2. Применение RL к дообучению моделей генерации последовательностей

RL позволяет обойти ограничения supervised learning: не требует эталонных ответов, устраняет distribution shift (модель учится на собственных выходах) и напрямую оптимизирует негладкие метрики (BLEU, CIDEr) или человеческие предпочтения.

**Формулировка задачи:**
Генерация рассматривается как процесс принятия решений, где состоянием $s$ является контекст $x$ и история сгенерированных токенов, действием $a$ — выбор следующего токена, а наградой $R(s,a)$ — итоговая оценка качества всей последовательности. Целевая функция:
$$J = \mathbb{E}_{s \sim d(s), a \sim \pi(a \mid obs(s))} \left[ R(s, a) \right]$$

**Policy Gradient и REINFORCE:**
Оптимизация идёт напрямую по параметрам политики $\pi_\theta$. Теоретический градиент:
$$\nabla J = \mathbb{E}_{s \sim d(s), a \sim \pi(a \mid obs(s))} \left[ \nabla \log \pi_\theta(a \mid s) Q(s, a) \right]$$
На практике $Q(s,a)$ неизвестна, поэтому используется Monte-Carlo оценка по полным сгенерированным сессиям (REINFORCE):
$$[\nabla_\theta J]_{mc} = \frac{1}{N} \sum_{i=1}^{N} R_\psi(x_i, y_i) \cdot \nabla_\theta \log \pi_\theta(y_i \mid x_i)$$
Основная проблема метода — **высокая дисперсия** градиента, так как награда $R(s,a)$ сильно зависит от случайности траектории.

**Снижение дисперсии: Advantage и SCST**
Для стабилизации из награды вычитается базовая линия $b(s)$, не зависящая от действия. Оптимальный выбор — функция ценности состояния $V(s)$, что даёт Advantage:
$$A(s, a) = R(s, a) - V(s)$$
Для снижения дисперсии применяется **Self-Critical Sequence Training (SCST)**: в качестве базовой линии используется награда той же модели, но работающей в режиме жадного инференса (как в продакшене). Это убирает необходимость обучать отдельного критика:
$$A(s, a) = R(s, a) - R(s, a_{inference}(s))$$
Обновление происходит только для действий, которые принесли награду выше, чем жадный baseline.

**Промышленный пайплайн (RLHF и DPO):**
Современные LLM дообучаются по схеме InstructGPT:
1. **SFT (Supervised Fine-Tuning):** модель обучается на качественных диалогах через cross-entropy.
2. **Reward Model:** отдельная сеть обучается предсказывать предпочтения людей на парах ответов.
3. **RL-оптимизация:** модель генерирует ответы, Reward Model выставляет баллы, и политика обновляется через PPO или REINFORCE, увеличивая вероятность высокооценённых ответов.

Для упрощения пайплина используется **Direct Preference Optimization (DPO)**, который заменяет явный RL-цикл на прямую оптимизацию логарифмического правдоподобия предпочтений:
$$L_{DPO} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log p_*(y_w \succ y_l \mid x) \right]$$
где $y_w$ и $y_l$ — выигравший и проигравший ответы соответственно. Метод сохраняет преимущества выравнивания по предпочтениям, но обучается стабильнее, так как сводится к стандартному supervised-лоссу.

RL превращает генерацию из задачи имитации датасета в задачу прямой оптимизации конечной метрики качества. SCST и DPO решают проблему высокой дисперсии градиентов, делая дообучение стабильным и вычислительно эффективным.